# Exploratory Data Analysis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import os
from pathlib import Path
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact
from scipy.stats import mannwhitneyu

In [ ]:
# Import processed data from csv
# Define the path to the processed data based on the current user
current_user = os.getlogin()
PROC_DIR = Path(rf"/home/{current_user}/local-share/processed")

# Read the processed data
data = pd.read_csv(PROC_DIR / "features_derived.csv") 

# For now, we use v2_combined.csv as processed data. When it becomes final, we can change this to the final processed data file.

## Analyse relation between time_enroll_to_start_weeks and drop-out

In [ ]:
# Add color legend for sdo5_degree_drop_out
plt.figure(figsize=(10, 6))
sns.histplot(data, x='time_enroll_to_start_weeks', hue='sdo5_degree_drop_out', bins=30, kde=True, multiple="stack")
plt.title('Distribution of Time from Enrollment to Start (weeks) by Degree Drop Out Status')
plt.xlabel('Time from Enrollment to Start (weeks)')
plt.ylabel('Frequency')
plt.legend(title='Degree Drop Out Status', labels=['Dropped Out', 'Not Dropped Out'])
plt.show()

# Test if time_enroll_to_start_weeks differs between dropout groups
no_dropout = data[data['sdo5_degree_drop_out'] == 0]['time_enroll_to_start_weeks'].dropna()
dropout = data[data['sdo5_degree_drop_out'] == 1]['time_enroll_to_start_weeks'].dropna()

statistic, p_value = mannwhitneyu(no_dropout, dropout, alternative='two-sided')
print(f"Mann-Whitney U test: U={statistic}, p={p_value}")

# Effect size (rank-biserial correlation)
n1, n2 = len(no_dropout), len(dropout)
effect_size = 1 - (2 * statistic) / (n1 * n2)
print(f"Effect size (rank-biserial): {effect_size}")

There is a significant relation between time to enrollment and drop-out (p << 0.001) where students who drop out tend to enroll closer to the start date of the degree (last-minute enrollments). 

## Analyse if drop-out is higher among students that study multiple degrees at the same time

In [ ]:
# What is the prevalence of students enrolled in multiple degrees?
multi_degree_prevalence = data['Multiple_Degrees_Enrollment'].mean()
print(f"Prevalence of multi-degree enrollment: {multi_degree_prevalence:.2%}")

# What is the drop-out rate for students with multiple degrees vs single degree?
dropout_rate_multi = data[data['Multiple_Degrees_Enrollment'] == 1]['sdo5_degree_drop_out'].mean()
dropout_rate_single = data[data['Multiple_Degrees_Enrollment'] == 0]['sdo5_degree_drop_out'].mean()
print(f"Drop-out rate for multi-degree students: {dropout_rate_multi:.2%}")
print(f"Drop-out rate for single-degree students: {dropout_rate_single:.2%}")

# Do multi-degree students have a significantly different drop-out rate compared to single-degree students?
contingency_table = pd.crosstab(data['Multiple_Degrees_Enrollment'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared test p-value: {p}")

# For multi-degree students, count the number of degrees that they drop out of (0, 1 or 2)
multi_degree_data = data[data['Multiple_Degrees_Enrollment'] == 1]
dropout_counts = multi_degree_data.groupby('sdo1_characteristics_student_number')['sdo5_degree_drop_out'].sum().value_counts().sort_index()
dropout_percentages = (dropout_counts / dropout_counts.sum()) * 100
dropout_counts = pd.DataFrame({'Count': dropout_counts, 'Percentage': dropout_percentages})
print("Dropout counts for multi-degree students:")
print(dropout_counts)

Students studying multiple degrees are uncommon within the HU among freshman fulltime bachelor students (0.5%). However, these students studying multiple degrees at the same time are at high-risk of dropping out of at least one of their degrees (97%, p < 0.001). In most cases a student drops out of one degree and continues with the other degree (60%), but in a large proportion of cases students are seen to drop out of both degrees (37%). 

## Analyse SKC relation with drop-out

In [ ]:
# Analyse relation of sdo2_skc_ADVIES_DEF with drop-out, and include missing data
# Include number of students who dropped out per SKC advice category
data['sdo2_skc_ADVIES_DEF'] = data['sdo2_skc_ADVIES_DEF'].fillna('Missing')
data['sdo5_degree_drop_out'] = data['sdo5_degree_drop_out'].fillna('Missing')
sdo_skc_dropout = data.groupby('sdo2_skc_ADVIES_DEF')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100

# Total number of students per SKC advice category, with percentages
print(data['sdo2_skc_ADVIES_DEF'].value_counts(normalize=True) * 100)

# First look at how many students dropped out per SKC advice category
drop_out_per_SKC_collegeyear = data.groupby('sdo2_skc_ADVIES_DEF')['sdo5_degree_COLLEGEJAAR'].value_counts().unstack().fillna(0)
print(drop_out_per_SKC_collegeyear)

# Visualize as a line graph: each advice type is a line over the college years
drop_out_per_SKC_collegeyear.T.plot(kind='line', marker='o', figsize=(10, 6))
plt.title('Aantal studenten per SKC advies over collegejaren')
plt.ylabel('Aantal studenten')
plt.xlabel('Collegejaar')
plt.legend(title='SKC Advies', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Some advice categories have very few students, exclude them from significant analysis
# Exclude studies with less than 100 students in that category
sdo_skc_counts = data['sdo2_skc_ADVIES_DEF'].value_counts()
valid_skc_categories = sdo_skc_counts[sdo_skc_counts >= 200].index
sdo_skc_dropout = sdo_skc_dropout.loc[valid_skc_categories]

# Print the drop-out rates by SKC advice category
print(sdo_skc_dropout)

# Visualize the results
sdo_skc_dropout.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by SKC Advice Category')
plt.ylabel('Percentage')
plt.xlabel('SKC Advies')
plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Are any of the differences significant?
contingency_table = pd.crosstab(data['sdo2_skc_ADVIES_DEF'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Exlude 'Missing' category for significance testing
sdo_skc_dropout = sdo_skc_dropout.drop(index='Missing', errors='ignore')

# Print the drop-out rates by SKC advice category
print(sdo_skc_dropout)

# Visualize the results
sdo_skc_dropout.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by SKC Advice Category')
plt.ylabel('Percentage')
plt.xlabel('SKC Advies')
plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Are any of the differences significant?
#Exclude categories with less than 100 students
data_no_small = data[data['sdo2_skc_ADVIES_DEF'].isin(valid_skc_categories)]

#Exclude 'Missing' category for significance testing
data_no_missing = data_no_small[data_no_small['sdo2_skc_ADVIES_DEF'] != 'Missing']
contingency_table = pd.crosstab(data_no_missing['sdo2_skc_ADVIES_DEF'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-squared: {chi2}, p-value: {p}")

### Results
SKC Advice type has a strong significant relationship with HU freshman drop-out (p << 0.001). Interestingly, a lot of students (10.7%) that start a degree do not have any SKC advice type (i.e. `Missing` advice type) and this group of students shows the lowest drop-out rates (23.3%). When we exclude this SKC advice type, the strong significant effect between SKC advice type and HU freshman drop-out remains (p << 0.001), indicating that the 'actual' SKC advice types also have a strong relation with drop-out. 

Highest drop-out was observed for students with a `Negatief` advice type (45.5 of students drop-out in their first year), followed by `Positief met aandachtspunten` and `Neutraal` (41.4% and 34.6% respectively). Lowest drop-out was observed in students that received a `Positief` advice type (32.1%).

## Analyse previous education school name with drop-out
Note that this analysis can no longer be ran since previous education school name has been excluded as a data field due to the possibility of indirect profiling. 

In [ ]:
# # Analyse relation of sdo1_previous_education with drop-out, and include missing data
# # Include number of students who dropped out per previous education category
# data['sdo1_previous_Previous_School'] = data['sdo1_previous_Previous_School'].fillna('Missing')
# print(data.groupby('sdo1_previous_Previous_School')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100)

# # Find all previous education categories with at least 100 students
# sdo_prev_counts = data['sdo1_previous_Previous_School'].value_counts()
# valid_prev_categories = sdo_prev_counts[sdo_prev_counts >= 100].index

# # Select only rows with previous education categories > 100 students
# data_prev_valid = data[data['sdo1_previous_Previous_School'].isin(valid_prev_categories)]
# # Exclude 'Missing' category for significance testing
# data_prev_valid = data_prev_valid[data_prev_valid['sdo1_previous_Previous_School'] != 'Missing']
# # Print the drop-out rates by previous education category
# print(data_prev_valid.groupby('sdo1_previous_Previous_School')['sdo5_degree_drop_out'].value_counts())

# # Use Fisher's Exact Test to determine if differences are significant, using filtered data
# contingency_table_prev = pd.crosstab(data_prev_valid['sdo1_previous_Previous_School'], data_prev_valid['sdo5_degree_drop_out'])
# oddsratio, p_value = fisher_exact(contingency_table_prev)
# print(f"Odds Ratio: {oddsratio}, p-value: {p_value}")

# # Group sdo1_previous_Previous_School into groups based group size
# def categorize_school_size(count):
#     if count < 10:
#         return 1
#     elif 10 <= count < 50:
#         return 2
#     elif 50 <= count < 100:
#         return 3
#     elif 100 <= count < 500:
#         return 4
#     else:
#         return 5
    
# school_size_counts = data['sdo1_previous_Previous_School'].value_counts()
# data_regrouped = data.copy()
# data_regrouped['Previous_School_Size'] = data['sdo1_previous_Previous_School'].map(school_size_counts).apply(categorize_school_size)
# contingency_table_prev = pd.crosstab(data_regrouped['sdo1_previous_Previous_School'], data_regrouped['sdo5_degree_drop_out'])

# # Calculate Chi square test
# chi2, p, dof, expected = chi2_contingency(contingency_table_prev)
# print(f"Chi-squared: {chi2}, p-value: {p}")

# contingency_table_size = pd.crosstab(data_regrouped['Previous_School_Size'], data_regrouped['sdo5_degree_drop_out'])

# # Test for trend using logistic regression
# X = sm.add_constant(data_regrouped['Previous_School_Size'])
# model = sm.Logit(data_regrouped['sdo5_degree_drop_out'], X).fit()
# print(model.summary())

# # Visualize the results
# drop_out_per_prevschool = data_regrouped.groupby('Previous_School_Size')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
# drop_out_per_prevschool.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by Previous Education School Size')
# plt.ylabel('Percentage')
# plt.xlabel('Previous Education School Size')
# plt.legend(title='Drop-out Status', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

### Results
There are many different (900) previous education schools (i.e. BRIN codes) in the dataset and a large number of schools only have a small number of students that start a degree at the HU: about 800 schools have seen less than 100 students starting a degree at the HU. We can conclude from this that most students that start a degree at the HU come from a relatively small subset of schools (i.e. 106 instead of 900 in our dataset). 

Since some schools have a small number of students, we've used Fisher's Exact Test for small sample sizes with a cut-off of at least 10 students per school. 

We found that there is a strong association between previous education school and HU freshman drop-out (Odds ratio ~ 1.40 * 10^-182, p = 0.0001), but after correcting for multiple comparisons this not statistically significant (p > 0.05). 

Since using previous education school might lead to profiling, combined with above multiple testing complexity, we grouped schools based on how many students they provided to the HU and tested: one group for less than 10 students, one group for between 10 and 50 students, one group for between 50 and 100 students, one group for betwen 100 and 500 students and one group for more then 500 students. 
We found a very strong signification association between previous education HU student size with respect to HU freshman drop-out (Chi-squared = 1282, p << 0.001). 

## Investigate missing SKC advice type

### Results
For about 11% students in the 'predicting HU drop-out' dataset (i.e. 2018-2023) there is no SKC advice available. When looking further, it seems that this is mostly true for a small number of HU degrees SKC data. For most of these degrees the SKC advice is never available. These degrees are numerus fixus degrees and do not have SKC. It would be worthwhile to include numerus fixus as a feature in the dataset. 

## Analyse relation between drop-out and DSA

In [ ]:
# Look at BSA/DSA (has_dsa column) effect on drop-out (sdo5_degree_drop_out)
sdo5_degree_drop_out_by_dsa = data.groupby('has_dsa')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
print(sdo5_degree_drop_out_by_dsa)

# Test for significance
contingency_table_dsa = pd.crosstab(data['has_dsa'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table_dsa)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Calculate effect size (Cramer's V)
n = contingency_table_dsa.values.sum()
rows, cols = contingency_table_dsa.shape
min_dim = min(rows - 1, cols - 1)
if n > 0 and min_dim > 0:
    cramers_v = (chi2 / (n * min_dim)) ** 0.5
else:
    cramers_v = float('nan')

# Helper to convert p-value to significance stars
def sig_stars(pval):
    if pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    else:
        return 'ns'

stars = sig_stars(p)

# Visualize the results and include chi-squared significance (annotate on plot)
ax = sdo5_degree_drop_out_by_dsa.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by BSA/DSA Status')
plt.ylabel('Percentage')
plt.xlabel('Has DSA')
plt.legend(title='Drop-out [1 = yes]', bbox_to_anchor=(1.05, 1), loc='upper left')
# wrap title
plt.title('HU full-time bachelor freshmen drop-out is significantly lower for DSA-degrees (p << 0.001), \n ' \
'but this effect is most likely due to confounding factors (Cramers V < 0.1).')
# Compose annotation text with formatted numbers
annot_text = f"Chi2={chi2:.2f}, p={p:.3g} {stars}\nCramer's V={cramers_v:.3f}\nDOF={dof}"
# Place the annotation in the figure (upper-left inside the figure canvas)
plt.gcf().text(0.02, 0.95, annot_text, ha='right', va='top', fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
plt.tight_layout()
plt.show()

## Analyse relation drop-out and NF

In [ ]:
# Look at NF (has_nf) effect on drop-out (sdo5_degree_drop_out)
sdo5_degree_drop_out_by_nf = data.groupby('has_nf')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
print(sdo5_degree_drop_out_by_nf)

# Test for significance
contingency_table_nf = pd.crosstab(data['has_nf'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table_nf)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Calculate effect size (Cramer's V)
n = contingency_table_dsa.values.sum()
rows, cols = contingency_table_dsa.shape
min_dim = min(rows - 1, cols - 1)
if n > 0 and min_dim > 0:
    cramers_v = (chi2 / (n * min_dim)) ** 0.5
else:
    cramers_v = float('nan')

# Visualize the results
ax = sdo5_degree_drop_out_by_nf.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by NF Status')
plt.ylabel('Percentage')
plt.xlabel('Has NF')
plt.legend(title='Drop-out [1 = yes]', bbox_to_anchor=(1.05, 1), loc='upper left')
# wrap title
plt.title('HU full-time bachelor freshmen drop-out is significantly lower for NF-degrees (p << 0.001), \n ' \
'but NF is most likely onle of many variables effecting drop-out (Cramers V < 0.1).')
# Compose annotation text with formatted numbers
annot_text = f"Chi2={chi2:.2f}, p={p:.3g}\nCramer's V={cramers_v:.3f}\nDOF={dof}"
# Place the annotation in the figure (upper-left inside the figure canvas)
plt.gcf().text(0.02, 0.95, annot_text, ha='right', va='top', fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
plt.tight_layout()
plt.show()

In [ ]:
# Read the processed data
data = pd.read_csv(PROC_DIR / "cleaned_original.csv") 

## Analyse drop-out relation among age groups

In [ ]:
# Exclude collegeyear 2018, 2019 and 2020 for age analysis (data may be incomplete)
data_age = data[~data['sdo5_degree_COLLEGEJAAR'].isin([2018, 2019, 2020])]

# Look at age distribution with 1 bin for every year
plt.figure(figsize=(10, 6))
data_age['sdo1_characteristics_age_start_study'].plot(kind='hist', bins=range(15, 31), rwidth=0.8)
plt.title('Age Distribution of Students at Start of Study')
plt.xlabel('Age')
plt.ylabel('Number of Students')
plt.xticks(range(15, 31))
plt.tight_layout()
plt.show()

# Average and median age
avg_age = data_age['sdo1_characteristics_age_start_study'].mean()
median_age = data_age['sdo1_characteristics_age_start_study'].median()
print(f"Average Age: {avg_age}, Median Age: {median_age}")

# Create above age groups
data_age['age_group'] = pd.cut(data_age['sdo1_characteristics_age_start_study'], bins=[0, 17, 19, 22, 100], labels=['17-', '18-19', '20-22', '23+'])

# Visualize group sizes
age_group_counts = data_age['age_group'].value_counts().sort_index()
age_group_counts.plot(kind='bar', figsize=(8, 5), title='Number of Students per Age Group')
plt.ylabel('Number of Students')
plt.xlabel('Age Group')
plt.tight_layout()
plt.show()

# Is there any significant difference in drop-out rates between age groups?
drop_out_by_age_group = data_age.groupby('age_group')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
print(drop_out_by_age_group)
# Test for significance
contingency_table_age = pd.crosstab(data_age['age_group'], data_age['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table_age)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Calulate effect size (Cramer's V)
n = contingency_table_age.values.sum()
rows, cols = contingency_table_age.shape
min_dim = min(rows - 1, cols - 1)
if n > 0 and min_dim > 0:
    cramers_v = (chi2 / (n * min_dim)) ** 0.5
else:
    cramers_v = float('nan')

In [ ]:
# Look at drop-out rates by age group
ax = drop_out_by_age_group.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by Age Group')
plt.ylabel('Percentage')
plt.xlabel('Age Group')
plt.legend(title='Drop-out [1 = yes]', bbox_to_anchor=(1.05, 1), loc='upper left')
# wrap title
plt.title('HU full-time bachelor freshmen drop-out rates by age group (p < 0.05), \n ' \
'but age group effect is small (Cramers V < 0.1).')
# Compose annotation text with formatted numbers
annot_text = f"Chi2={chi2:.2f}, p={p:.3g} \nCramer's V={cramers_v:.3f}\nDOF={dof}"
# Place the annotation in the figure (upper-left inside the figure canvas)
plt.gcf().text(0.02, 0.95, annot_text, ha='right', va='top', fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
plt.tight_layout()
plt.show()

### Results & conclusions
 From a business / subject matter perspective we would like to bin the student ages into 'early starters', 'common age group', 'late starters' and 'career changers'. Median and average age of the population is 19.0 and 19.3 years respectively, so most likely one of the middle bins needs to include this age. When looking at the age distribution there is a large proportion of students younger than 17 years old, so it makes sense to bin this group into 'early starters' (i.e. age < 17 years old). Looking at the right end of the age distribution there is somewhat of a tail (left-skewed distribution), indicating that the 'career changers' might have a lower starting age than expected to have a sufficient group size. To make 4 reasonably similarly sized groups with above considerations in mind we have chosen the following:
 - < 17   (early starters)
 - 17-18 (common age group)
 - 19-21 (late starters)
 - '> 21   (career changers)

 We found that the relation between these age groups and drop-out is very significant (p << 0.001), whereas drop-out is highest among the career changers group (37%). The effect of this relation is very small (Cramer's V = 0.03) indicating that this is only one small factor influencing drop-out. 

## Analyse relation between drop-out and travel distance

In [ ]:
# Look at distance to campus effect on drop-out (sdo1_postal_distance_distance_to_3584_CS) using logistic regression
# Remove rows with missing distance or drop-out data
data_distance = data.dropna(subset=['sdo1_postal_distance_distance_to_3584_CS', 'sdo5_degree_drop_out'])
import statsmodels.api as sm
X = sm.add_constant(data_distance['sdo1_postal_distance_distance_to_3584_CS'])
y = data_distance['sdo5_degree_drop_out']
log_reg_model = sm.Logit(y, X).fit()
print(log_reg_model.summary())

# Look at distribution of distances
plt.figure(figsize=(10, 6))
data['sdo1_postal_distance_distance_to_3584_CS'].plot(kind='hist', bins=50, title='Distribution of Distance to HU')
plt.xlabel('Distance to HU (km)')
plt.ylabel('Number of students')
plt.tight_layout()
plt.show()

# Group distances into categories
# 'close' (0-10 km), 'medium' (10-80 km), 'far' (80+ km)
data['distance_bin'] = pd.cut(data['sdo1_postal_distance_distance_to_3584_CS'], bins=[0, 10, 80, float('inf')], labels=['close', 'medium', 'far'], right=False)

# Look at drop-out rates by distance bin
drop_out_by_distance_bin = data.groupby('distance_bin')['sdo5_degree_drop_out'].value_counts(normalize=True).unstack().fillna(0) * 100
print(drop_out_by_distance_bin)

# Test for significance
contingency_table_distance = pd.crosstab(data['distance_bin'], data['sdo5_degree_drop_out'])
chi2, p, dof, expected = chi2_contingency(contingency_table_distance)
print(f"Chi-squared: {chi2}, p-value: {p}")

# Visualize the results
ax = drop_out_by_distance_bin.plot(kind='bar', stacked=True, figsize=(10, 6), title='Drop-out Rate by travel distance to HU')
plt.ylabel('Percentage')
plt.xlabel('Distance Bin')
plt.legend(title='Drop-out [1 = yes]', bbox_to_anchor=(1.05, 1), loc='upper left')
# wrap title
plt.title('HU full-time bachelor freshmen drop-out rates by travel distance to HU (p > 0.05).')
# Compose annotation text with formatted numbers  
annot_text = f"Chi2={chi2:.2f}, p={p:.3g}"
# Place the annotation in the figure (upper-left inside the figure canvas)
plt.gcf().text(0.02, 0.95, annot_text, ha='right', va='top', fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
plt.tight_layout()
plt.show()

### Results & Conclusions
There is no significant relation to the distance between a student's postal address and the HU (travel distance) and drop-out (logistic regression, p > 0.05). 